In [ ]:
# imports and stuff
import numpy as np
import pandas as pd

In [ ]:
# load in the base dataframes we will be working with
# YOU WILL NEED TO UPDATE THE PATH TO THESE FILES!!
base_rent = pd.read_excel('raw datasets/FY2026-Rent.xlsx')
base_income = pd.read_csv('raw datasets/MedianIncomeData/MedianIncomeData.csv')
base_crime = pd.read_csv('raw datasets/ba_crime_combined.csv')
base_pop = pd.read_csv('raw datasets/CA_county_pop.csv')

In [ ]:
# new database with better names. Suffixes _90 and _110 denote their margin of error (RENT_STUDIO_90) 
# is the lower end of RENT_STUDIO, RENT_STUDIO_110 is the upper end. #BD refers to number of bedrooms.

rent = base_rent.rename(columns={
    'ZIP\nCode': "ZIP_CODE",
    'HUD Area Code': "HUD_AREA_CODE",
    'HUD Fair Market Rent Area Name': 'CITY_STATE',
    'SAFMR\n0BR': 'RENT_STUDIO',
    'SAFMR\n0BR - 90%\nPayment\nStandard': 'RENT_STUDIO_90',
    'SAFMR\n0BR - 110%\nPayment\nStandard': 'RENT_STUDIO_110',
    'SAFMR\n1BR': 'RENT_1BD',
    'SAFMR\n1BR - 90%\nPayment\nStandard': 'RENT_1BD_90',
    'SAFMR\n1BR - 110%\nPayment\nStandard': 'RENT_1BD_110',
    'SAFMR\n2BR': 'RENT_2BD',
    'SAFMR\n2BR - 90%\nPayment\nStandard': 'RENT_2BD_90',
    'SAFMR\n2BR - 110%\nPayment\nStandard': 'RENT_2BD_110',
    'SAFMR\n3BR': 'RENT_3BD',
    'SAFMR\n3BR - 90%\nPayment\nStandard': 'RENT_3BD_90',
    'SAFMR\n3BR - 110%\nPayment\nStandard': 'RENT_3BD_110',
    'SAFMR\n4BR': 'RENT_4BD',
    'SAFMR\n4BR - 90%\nPayment\nStandard': 'RENT_4BD_90',
    'SAFMR\n4BR - 110%\nPayment\nStandard': 'RENT_4BD_110'
})

In [ ]:
# get only california
rent = rent[rent['CITY_STATE'].str.contains(', CA')]

In [ ]:
# get only Bay Area
rent = rent[rent['CITY_STATE'].isin([
    "Oakland-Fremont, CA HUD Metro FMR Area",
    "San Francisco, CA HUD Metro FMR Area",
    "San Jose-Sunnyvale-Santa Clara, CA HUD Metro FMR Area",
    "Napa, CA MSA",
    "Santa Rosa-Petaluma, CA MSA",
    "Vallejo, CA MSA"])]

In [ ]:
# rent for different types of living spaces in the Bay Area is now in the variable 'rent'
# Finally, we change ZIP_CODE to be of type str instead of int for general ease.
rent["ZIP_CODE"] = rent["ZIP_CODE"].astype(str)
# for a specific example, here's rent for zip codes in the general San Francisco area:
rent[rent["CITY_STATE"].str.contains("San Francisco")].head()

In [ ]:
# NOW LOOKING AT BASE MEDIAN INCOME

In [ ]:
# Now we set up our database for the median income. Before we merge the zip codes with the rent database to
# account only for Bay Area zip codes, we gotta do some cleaning up. We set a column just for zip codes,
# rename the columns to something readable, then filter out the unnecessary columns. This new dataframe
# is called income as opposed to base_income.
income = base_income
income['ZIP_CODE'] = base_income['NAME'].str.split().str[1]
income = income.rename(columns={
    'S1901_C01_001E': "TOTAL_HOUSEHOLDS_EST",
    'S1901_C01_012E': "MEDIAN_INCOME_HOUSEHOLD_EST"
})
income = income.loc[2:, ["GEO_ID", "ZIP_CODE", "MEDIAN_INCOME_HOUSEHOLD_EST", "TOTAL_HOUSEHOLDS_EST"]]

In [ ]:
# Now we merge, specifically an inner merge with rent's already determined Bay Area zip codes.
# We DO NOT WANT zip codes that do not pertain to housing, like SFO Airport (94128) or PO Box
# zip codes. Inner merge accounts for this by making sure the zip code has both people living
# there, paying rent, and having enough households to measure median income. Of course, income
# still has missing values (represented by the hyphen: -) that are exclusive to the Bay Area,
# like the aforementioned SFO Airport. We will drop them after our merge. Them having < 200
# TOTAL_HOUSEHOLDS_EST is a red flag already, and those with high numbers tend to be AirBNB
# or hotel areas on large natural plots of land (e.g. 94923). Not what we're measuring with 
# this project. So, we remove those and any zip codes with fewer than 200 estimated households.
# The > 200 estimated households filter only removes 11 zip codes.

ba_data = rent.merge(income, on="ZIP_CODE", how="inner")
ba_data = ba_data[ba_data["MEDIAN_INCOME_HOUSEHOLD_EST"] != "-"]
ba_data = ba_data[ba_data["TOTAL_HOUSEHOLDS_EST"].astype(int) > 200]

In [ ]:
# NOW LOOKING AT CRIME

In [ ]:
# First things first, we pull population data per county to get a 
# measurement for crime rates. We filter down to only Bay Area
# counties. Then, we keep only years with entry '1' or '5' (2020 and 2024 approximately). We
# can use these to measure the crime rates for each year and see if crime is increasing or not.
pop = base_pop[base_pop["CTYNAME"].isin(
    ["Alameda County",
    "Contra Costa County",
    "Marin County",
    "Napa County",
    "San Francisco County",
    "San Mateo County",
    "Santa Clara County",
    "Solano County",
    "Sonoma County",]
)]
pop = pop[(pop["YEAR"] == 1) | (pop["YEAR"] == 5)] #keep pop stats in 2020 and 2024
pop = pop[["CTYNAME", "YEAR", "POPESTIMATE", "MEDIAN_AGE_TOT"]] #keep the columns we care about
pop

In [ ]:
# Now, we separate violent crime from property crime based on county. We can fix the entries in
# the COUNTY columns to make a future join with the pop database easier.

base_crime = base_crime.replace({
    "Santaclara": "Santa Clara",
    "Contracosta": "Contra Costa",
    "Sanfrancisco": "San Francisco",
    "Sanmateo": "San Mateo",
})
violent_crime = base_crime[base_crime['CRIME_CATEGORY'] == "Violent Crimes"]
violent_crime.loc[:, "2020"] = violent_crime["2020"].str.replace(",", "").astype(int) #turn values to int
violent_crime.loc[:, "2024"] = violent_crime["2024"].str.replace(",", "").astype(int) #turn values to int
property_crime = base_crime[base_crime['CRIME_CATEGORY'] == "Property Crimes"]
property_crime.loc[:, "2020"] = property_crime["2020"].str.replace(",", "").astype(int) #turn values to int 
property_crime.loc[:, "2024"] = property_crime["2024"].str.replace(",", "").astype(int) #turn values to int

In [ ]:
# We clean the population table up a bit before 
pop["CTYNAME"] = pop["CTYNAME"].str.split(" County").str[0]
pop = pop.rename(columns={"CTYNAME":"COUNTY"})

In [ ]:
# And finally, clearing up what the ambiguous 'year column' represents
pop = pop.rename(columns={"YEAR":"POPYEAR"})
pop["POPYEAR"] = pop["POPYEAR"].replace({1: 2020, 5: 2024})

In [ ]:
# Now we merge population and crime, before using pivot_tables to get our year-based crime and
# population data side by side for each county.

violent_df = pop.merge(violent_crime, on="COUNTY")
property_df = pop.merge(property_crime, on="COUNTY")

# We now create a 'wide' population table using pivot_table.
# This reshapes the data, turning the values from 'POPYEAR' (2020, 2024) into new columns.
pop_df = property_df.pivot_table(
    index='COUNTY', 
    columns='POPYEAR', 
    values='POPESTIMATE'
).rename(columns={2020: 'POP2020', 2024: 'POP2024'})

# Merge the population data, drop duplicates, and calculate rates in one chained command.
property_df = property_df.merge(pop_df, on='COUNTY') \
                      .drop_duplicates(subset=['COUNTY', 'CRIME_CATEGORY'])
property_df

#---------

# We do the same for the violent_df
vi_df = violent_df.pivot_table(
    index='COUNTY', 
    columns='POPYEAR', 
    values='POPESTIMATE'
).rename(columns={2020: 'POP2020', 2024: 'POP2024'})

#  Merge the population data, drop duplicates, and calculate rates in one chained command.
violent_df = violent_df.merge(vi_df, on='COUNTY') \
                      .drop_duplicates(subset=['COUNTY', 'CRIME_CATEGORY'])
violent_df

In [ ]:
# Now, violent_df and property_df accurately show population and crime data per year 2020 and 2024
# Theoretically, we could make it clearer that columns 2020 and 2024 pertain to crime, but since
# we only care about crime rates for our purposes, I won't bother. We'll also shorten crime_category
# to just say the type of crime, not the word 'crimes'

violent_df["2020_CRIMERATE_VIOL"] = violent_df["2020"] / violent_df["POP2020"]
violent_df["2024_CRIMERATE_VIOL"] = violent_df["2024"] / violent_df["POP2024"]
violent_df["CHANGE_IN_CRIME_VIOL%"] = violent_df["2024_CRIMERATE_VIOL"] / violent_df["2020_CRIMERATE_VIOL"] * 100
violent_df["CRIME_CATEGORY"] = violent_df["CRIME_CATEGORY"].str.split().str[0]

property_df["2020_CRIMERATE_PROP"] = property_df["2020"] / property_df["POP2020"]
property_df["2024_CRIMERATE_PROP"] = property_df["2024"] / property_df["POP2024"]
property_df["CHANGE_IN_CRIME_PROP%"] = property_df["2024_CRIMERATE_PROP"] / property_df["2020_CRIMERATE_PROP"] * 100
property_df["CRIME_CATEGORY"] = property_df["CRIME_CATEGORY"].str.split().str[0]

In [ ]:
# Finally, let's merge violent_df and property_df into one final crime dataframe.
# First, drop unnecessary columns entirely.

violent_clean = violent_df[['COUNTY', '2020_CRIMERATE_VIOL', '2024_CRIMERATE_VIOL', 'CHANGE_IN_CRIME_VIOL%']]
property_clean = property_df[['COUNTY', '2020_CRIMERATE_PROP', '2024_CRIMERATE_PROP', "CHANGE_IN_CRIME_PROP%"]]

# Step 2: Merge the two clean DataFrames on the 'COUNTY' column.
# Since both tables have a 'COUNTY' column, this is very easy.
# Voila, crime_df accounts for crime based on County and measures its change over time, as well as whether its violent or property-related
crime_df = pd.merge(violent_clean, property_clean, on='COUNTY')
crime_df

In [ ]:
# NOW WE EXPORT!
# We can merge these cleaned dataframes in a separate jupyter file. As for our BART and CalTrain location info,
# that is recorded in latitude and longitude coordinates, not zip codes, so we can work with it in GeoPandas
# without necessarily needing to do any cleaning with it here.

ba_data.to_csv('cl_rent_income.csv', index=False) 
crime_df.to_csv('cl_crime.csv', index=False)